# 시간 설정

In [ ]:
import time
import random
import os

os.environ['TZ'] = 'Asia/Seoul'
time.tzset()

random.seed(1490)

def print_time() -> None:
    print(time.strftime('%Y년 %m월 %d일 %A %p %I시 %M분 %S초'))
    return

def timestamp() -> str:
    return time.strftime('%Y-%m-%d-%H-%M-%S')

print_time()
print(timestamp())

# 라이브러리

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt  # 그래프 보기
import seaborn as sns  # 그래프 보기

print_time()

# 전역변수

In [ ]:
global drops

# ROOT_PREFIX = "/content/drive/MyDrive/lgaimers5/" # 구글 드라이브용 루트
# ROOT_DIR = ROOT_PREFIX + "data" # 에이머스 제출 시 주석 처리 필요
ROOT_DIR = "data" # 구글 드라이브 실행 시 주석 처리 필요
save_dir = 'stats/'

RANDOM_STATE = 110 # 랜덤 시드
corr_thresh = 0.001 # 상관관계 삭제 기준
empty_thresh = 30 # 결측값 삭제 기준 비율

# 랜덤 시드 설정
np.random.seed(RANDOM_STATE)

# 1. 필터 1번, 필터 2번 등과 같이 순서가 있는 값은 추가 처리 없이 넘버링,
# 그 외 기타 order들도 같이 넘버링
numbering = ['Equipment_Dam', 'Model.Suffix_Dam', 'Workorder_Dam', \
             'Model.Suffix_AutoClave', 'Workorder_AutoClave', 'Equipment_Fill1', \
             'Model.Suffix_Fill1', 'Workorder_Fill1', 'Equipment_Fill2', \
             'Model.Suffix_Fill2', 'Workorder_Fill2']

# 2. OK, NG와 같이 범주형인 경우 가능하면 원 핫 인코딩
onehot = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', \
          'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2', \
          'Chamber Temp. Judge Value_AutoClave']

# 3. 숫자 값이 들어가야 하는데 OK가 혼입된 값임
custom = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', \
          'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

drops = [] # 값이 없거나 무의미해서 버릴 column들
dt_nan = []  # attributes to fill the missing value with any value
dt_del = []  # attributes to be deleted because there are too many missing values
need_general = []  # normalization list
del_list = [] # 상관관계 삭제 리스트
reduce_list = [] # 다중공선성 삭제 리스트
# 다중공선성: 서로 다른 속성끼리 깊은 연관을 갖는 것.
# 예를 들면 타겟은 따로 있는데 A 속성과 B 속성 둘 사이의 상관관계 값의 절댓값이 1에 가깝게 나오는 경우를 말함

print_time()

# 함수 정의

In [ ]:
import os
import shutil

def delete_all_files_in_folder(folder_path):
    # 폴더 내의 모든 파일과 폴더를 가져옵니다.
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        try:
            # 파일이면 삭제
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            # 폴더이면 폴더 자체를 삭제
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f'Error deleting {file_path}. Reason: {e}')
    return

print_time()

## 데이터 전처리

In [ ]:
def print_col_uniqs(data):
    global drops
    """
    unique 개수가 1개 이하인 column 삭제
    """
    init_cols = len(data.columns)
    total_cnt = len(data)

    for col in data.columns:
        if col == 'target':
            continue
        uniqs = data[col].unique()
        nuq = data[col].nunique()
        vcnt = data[col].value_counts()
        if (nuq <= 1 and len(uniqs) <= 1) or (len(data) == data[col].isna().sum()):
            drops.append(col)
        else:
            if nuq <= 10:
                # print(f'{col} : {nuq}, {uniqs}')
                pass
            else:
                # print(f'{col} : {nuq}')
                pass
            # 결측값 비율 보기(null이 있는 속성만)
            cnt = data[col].isna().sum()

            if cnt == 0:
                continue

            num = float(cnt) / float(total_cnt)

            if num * 100.0 <= empty_thresh:
                dt_nan.append(col)
            else:
                dt_del.append(col)
    
    drops += dt_del
    print(f'drops: {len(drops)}\ndt_del: {len(dt_del)}')
    data.drop(drops, axis=1, inplace=True)
    data.drop_duplicates(keep='first', inplace=True, ignore_index=True) # 중복 제거
    data[list(data.columns)].dropna(axis=0, inplace=True) # 모든 속성이 결측값인 행 제거
    
    # data.fillna(-1, inplace = True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum()}')
    print(f'num of initial cols: {init_cols}')
    print(f'num of deleted cols: {len(drops)}')
    print(f'num of remained cols: {len(data.columns)}')
    return data, drops

def print_columns_to_list(data):
    """
    컬럼이 너무 많아 바로 출력이 안되니 직접 출력
    """
    cnt = {}
    for col in data.columns:
        dt = str(data[col].dtypes)
        nu = data[col].nunique()
        if dt in cnt.keys():
            cnt[dt] += 1
        else:
            cnt[dt] = 1
        print(f'* {col} : {dt}({nu})')
    print(f'\ninfo: {cnt}')

def object_unique(data):
    """
    dtype이 object로 분류된 컬럼의 unique 값 확인
    """
    idx = 1
    for col in data.columns:
        if str(data[col].dtypes) == 'object':
            print(f'{idx}. {col}\n\t{data[col].nunique()}', end='')
            if data[col].nunique() <= 20:
                print(f', {data[col].unique()}')
            else:
                print()
            idx += 1

def enc_numbering(data, cols):
    """
    단순 넘버링으로 라벨 인코딩
    """
    print('넘버링')
    for col in cols:
        if col not in data.columns:
            continue
        data[col] = data[col].astype(str)
        le = LabelEncoder()
        le.fit(sorted(list(data[col].unique())))
        data[col] = le.transform(data[col])
    return data

def enc_onehot(data, cols):
    """
    원 핫 인코딩
    """
    print('원 핫 인코딩')
    for c in cols:
        if c not in data.columns:
            cols.remove(c)
    data = pd.get_dummies(data, columns = cols)
    return data

def enc_only_obj(data, cols):
    """
    다른 컬럼의 OK가 혼입된 값 정리, 소수 값을 리터럴 소수로 변환 후 dtype도 소수로 지정
    """
    print('문자열 숫자를 숫자로')
    for col in cols:
        if col not in data.columns:
            continue
        idx = 1
        uniqs = data[col].unique()
        for u in uniqs:
            try:
                data.loc[data[col] == u, col] = float(u)
            except:
                if u == 'OK' or u == -1:
                    data.loc[data[col] == u, col] = -1
                else:
                    data.loc[data[col] == u, col] = idx
                    idx += 1
        data[col] = data[col].astype(float)
        # print(data[col])
    return data

def obj_encoding(data, mode = 'train'):
    """
    모든 라벨 인코딩 실행
    """
    # 기본 넘버링
    data = enc_numbering(data, numbering)
    # 원 핫 인코딩
    # data = enc_onehot(data, onehot)
    data = enc_numbering(data, onehot)
    # OK가 혼입된 값 정리
    data = enc_only_obj(data, custom)
    # target 인코딩
    # 있는 경우에만
    if mode == 'train':
        print('target 값을 숫자로')
        data.loc[data['target'] == 'AbNormal', 'target'] = 1
        data.loc[data['target'] == 'Normal', 'target'] = 0
    return data

def show_heatmap(data, cnt = 12):
    """
    컬럼을 지정된 수만큼 묶어서 히트맵 출력
    * 색깔 참고: https://colorbrewer2.org/
    """
    cols = list(data.columns)
    corr = data.corr()
    print(corr.info())

    # plt.figure(figsize=(15, 10))
    # sns.heatmap(data.corr(), annot=True, cmap="RdBu", vmax=1)
    # plt.show()

def del_by_corr(data, thresh = 0.5, del_list = []):
    """
    지정된 값을 기준으로 상관관계가 낮은 컬럼 삭제
    """
    corr = data.corr()
    target_corr = corr['target']
    target_corr.drop('target', inplace=True)
    del_list = list(target_corr[target_corr.isnull() == True].index)
    temp = target_corr[-thresh < target_corr]
    del_list += list(temp[temp < thresh].index)

    data.drop(del_list, axis=1, inplace=True)
    print(f'num of cols deleted by correlation: {len(del_list)}')
    print(f'remain cols: {len(data.columns)}')

    return data, del_list

def corr_reduce(data, thresh=0.9, reduce_list=reduce_list):
    # 상관계수 행렬 계산
    correlation_matrix = data.corr().abs()

    # 상관관계 계수가 threshold 이상인 피처들 중 하나를 제거하기 위한 마스크 생성
    upper_triangle = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )

    # target_column과 연관된 피처를 제외하기 위해, target_column과 관련된 피처를 식별
    related_to_target = correlation_matrix['target'].index

    # 제거할 피처 리스트 생성
    to_drop = [column for column in upper_triangle.columns
               if column != 'target' and any(upper_triangle[column] > thresh)]

    # 상관관계가 높은 피처 제거
    df_reduced = data.drop(columns=to_drop)

    print("제거된 속성 수:", len(to_drop))

    return df_reduced, to_drop

def pre_processing(data, del_list=del_list, reduce_list=reduce_list):
    """
    데이터 전처리 통합 실행
    """
    # 데이터 뒤섞기
    print(f'# 데이터 뒤섞기')
    data = data.sample(frac=1)  # randomize data

    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    data, drops = print_col_uniqs(data)

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'train')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    data, del_list = del_by_corr(data, thresh=corr_thresh, del_list=del_list)
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data, reduce_list = corr_reduce(data)

    print('target 값 정수화')
    data['target'] = data['target'].astype(int)

    # 전처리 완료
    print('# 전처리 완료')
    print_time()

    return data, drops, del_list, reduce_list

def pre_processing_test(data):
    """
    테스트 데이터(레이블 X) 전처리 통합 실행
    """
    # unique 개수가 1개 이하인 column 삭제
    print('# unique 개수가 1개 이하인 column 삭제')
    print('속성 삭제')
    data.drop(drops, axis=1, inplace=True)
    for col in data.columns:
        if str(data[col].dtypes) != 'object':
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col].fillna(-1, inplace = True)
    print(f'결측값 처리 후 결측값 개수: {data.isna().sum().sum() - data["target"].isna().sum()}')

    # 모든 라벨 인코딩 실행
    print('# 모든 라벨 인코딩 실행')
    data = obj_encoding(data, 'test')

    # 상관관계가 낮은 컬럼 삭제
    print('# 상관관계가 낮은 컬럼 삭제')
    print(f'num of cols deleted by correlation: {len(del_list)}')
    data.drop(del_list, axis=1, inplace=True)
    print(f'remain cols: {len(data.columns)}')
    
    # 다중공선성이 높은 컬럼 삭제
    print('# 다중공선성이 높은 컬럼 삭제')
    data = data.drop(columns=reduce_list)

    # 테스트 데이터 전처리 완료
    print('# 테스트 데이터 전처리 완료')
    print_time()

    return data

print_time()

## 데이터 분석

In [ ]:
def col_stats(data, col):
    """
    특정 컬럼의 기초 통계 출력
    """
    normal = data[data['target'] == 0][col]
    abnormal = data[data['target'] == 1][col]

    cover = "*" * (len(col) + len("Name: , dtype: object"))
    stats = data[col].describe()
    nuq = data[col].nunique()
    
    for idx in stats.index:
        stats[idx] = round(stats[idx], 1)
    stats['name'] = col
    stats['n_unique'] = nuq
    stats['range'] = stats['max'] - stats['min']
    
    stats = stats.reindex(['name', 'n_unique', 'count', 'range', 'mean', 'std', 'min', '25%', '50%', '75%', 'max'])
    stats.drop(['25%', '75%'], inplace=True)
    stats = stats.rename({'50%': 'median'})
    
    return stats

def print_stats(data):
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    
    # 출력 내용을 저장할 파일 경로 지정
    output_file = save_dir + 'stats.txt'
    
    # 기존 stdout 백업
    original_stdout = sys.stdout

    normal = data[data['target'] == 0]
    abnormal = data[data['target'] == 1]
    
    # stdout을 파일로 리다이렉트
    with open(output_file, 'w') as f:
        sys.stdout = f
    
        for col in data.columns:
            d1 = col_stats(data, col)
            d1['name'] = 'total'
            d2 = col_stats(normal, col)
            d2['name'] = ' normal'
            d3 = col_stats(abnormal, col)
            d3['name'] = ' abnormal'
            stats = pd.concat([d1, d2, d3], axis=1)
            stats.columns = ['', '', '']

            sys.stdout = f
            print('*' * max(37, len(col)))
            print(col)
            print(stats)
            print('*' * max(37, len(col)))
            print()
            
            sys.stdout = original_stdout
            print('*' * max(37, len(col)))
            print(col)
            print(stats)
            print('*' * max(37, len(col)))
            print()
    sys.stdout = original_stdout

    return

print_time()

In [ ]:
def print_col_hist(data, col):
    """
    특정 컬럼의 히스토그램 출력
    """
    if not os.path.exists(save_dir + 'hist/'):
        os.mkdir(save_dir + 'hist/')
        print('dir maked')
    
    normal = data[data['target'] == 0][col]
    abnormal = data[data['target'] == 1][col]
    
    plt.figure(figsize=(15, 7))
    
    plt.subplot(1, 2, 1)
    sns.histplot(
        data[col],
        bins='fd', # 'auto', 'fd', 'doane', 'scott', 'stone', 'rice', 'sturges', or 'sqrt'
                    # https://numpy.org/doc/stable/reference/generated/numpy.histogram_bin_edges.html#numpy.histogram_bin_edges
        kde=True,
        stat='percent', # count, frequency, probability, percent, density
    )
    plt.title(f'{col} total')
    
    plt.subplot(1, 2, 2)
    sns.histplot(
        abnormal,
        bins='fd', # 'auto', 'fd', 'doane', 'scott', 'stone', 'rice', 'sturges', or 'sqrt'
                    # https://numpy.org/doc/stable/reference/generated/numpy.histogram_bin_edges.html#numpy.histogram_bin_edges
        kde=True,
        stat='percent',
        color=sns.color_palette("Set2")[1]
    )
    
    sns.histplot(
        normal,
        bins='fd', # 'auto', 'fd', 'doane', 'scott', 'stone', 'rice', 'sturges', or 'sqrt'
                    # https://numpy.org/doc/stable/reference/generated/numpy.histogram_bin_edges.html#numpy.histogram_bin_edges
        kde=True,
        stat='percent',
        color=sns.color_palette("Set2")[0],
        alpha=0.3
    )
    
    plt.title(f'{col} normal and abnormal')
    plt.legend(['normal', 'abnormal'])
    plt.savefig(save_dir + f'hist/hist_{col}.png', bbox_inches='tight')
    plt.show()
    return

def print_hist(data):
    if not os.path.exists(save_dir + 'hist/'):
        os.mkdir(save_dir + 'hist/')
        print('dir maked')
    else:
        delete_all_files_in_folder(save_dir + 'hist/')
        print('dir cleared')
    
    for col in data.columns:
        print(f'{col}')
        print_col_hist(data, col)
        print()
    return

print_time()

# 분석 실행

## 데이터 준비

In [ ]:
print_time()

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))

# 데이터 전처리
train_data, drops, del_list, reduce_list = pre_processing(train_data)

In [ ]:
print(f'drops: {len(drops)}, del_list: {len(del_list)}, reduce_list: {len(reduce_list)}')

## 기초 통계

In [ ]:
train_data.info()

In [ ]:
print_stats(train_data)

## 히스토그램

In [ ]:
print_hist(train_data)

## 전체 히트맵

In [ ]:
plt.figure(figsize=(100, 100))
sns.heatmap(train_data.corr(), annot=True, cmap="RdBu", vmax=1, vmin=-1)
plt.savefig(save_dir + 'corr_all.png', bbox_inches='tight')
plt.show()

## 부분 히트맵

In [ ]:
if not os.path.exists(save_dir + 'corr/'):
    os.mkdir(save_dir + 'corr/')
    print('dir maked')
else:
    delete_all_files_in_folder(save_dir + 'corr/')
    print('dir cleared')

corr_matrix = train_data.corr()

# 하삼각 행렬 부분만 추출하기
mask = np.tril(np.ones_like(corr_matrix, dtype=bool))

# 히트맵을 12x12로 나눠서 출력 (하삼각 부분만)
n_sections = 12
section_size = len(train_data.columns) // n_sections

# 히트맵 구역 나누기 (하삼각 부분만 출력)
for i in range(n_sections):
    for j in range(i + 1):  # j는 i보다 크지 않음 (하삼각 영역)
        plt.figure(figsize=(10, 8))
        section = corr_matrix.iloc[i * section_size:(i + 1) * section_size, j * section_size:(j + 1) * section_size]
        section_mask = mask[i * section_size:(i + 1) * section_size, j * section_size:(j + 1) * section_size]
        sns.heatmap(section, mask=~section_mask, annot=True, cmap="RdBu", vmax=1, vmin=-1)
        plt.savefig(save_dir + f'corr/corr_{i * section_size}_{(i + 1) * section_size}_{j * section_size}_{(j + 1) * section_size}.png', bbox_inches='tight')
        plt.show()